# APEX Optimizer Tutorial: Math Problem Solving

This tutorial demonstrates how to use **APEX** (Adaptive Prompt EXploration) to improve GPT-4 Mini's performance on AIME math problems through systematic prompt optimization.

APEX analyzes failures, recognizes success patterns, generates hypotheses, and validates improvements empirically.

## Configuration

All modifiable parameters in one place for easy adjustment:

In [1]:
# API Configuration
api_key = 'sk-12345'# Will prompt if not set
base_url = 'https://nexus-master.lmndstaging.com'

# Student Model Configuration (model being optimized)
student_model = "litellm_proxy/openai/gpt-5-mini"
student_base_url = base_url  # Optional custom API endpoint
student_temperature = 0.0  # Deterministic for math
student_reasoning_effort = 'minimal'

# Analysis Model Configuration (for failure analysis and hypotheses)
analysis_model = "litellm_proxy/openai/gpt-5"
analysis_base_url = base_url  # Optional custom API endpoint
analysis_temperature = 1.0  # Creative for hypothesis generation
analysis_reasoning_effort = 'minimal'

# APEX Optimization Settings
max_iterations = 3
num_hypotheses = 2
train_sample_size = 20
success_threshold = 1.0
convergence_patience = 2
num_threads = 50
seed = 42

# MLflow Tracking (Optional)
use_mlflow = True
mlflow_tracking_uri = "http://localhost:5005"
mlflow_experiment_name = "APEX-AIME-Math"

## Setup

Import dependencies and configure language models:

In [2]:
import os
import dspy
from dspy.adapters import JSONAdapter

if api_key is None:
    api_key = os.getenv("OPENAI_API_KEY") or input("Enter your OpenAI API key: ")

# Configure student model
student_kwargs = {
    "model": student_model,
    "api_key": api_key,
    "temperature": student_temperature,
}
if student_base_url:
    student_kwargs["base_url"] = student_base_url
if student_reasoning_effort:
    student_kwargs["reasoning_effort"] = student_reasoning_effort

student_lm = dspy.LM(**student_kwargs)

# Configure analysis model
analysis_kwargs = {
    "model": analysis_model,
    "api_key": api_key,
    "temperature": analysis_temperature,
}
if analysis_base_url:
    analysis_kwargs["base_url"] = analysis_base_url
if analysis_reasoning_effort:
    analysis_kwargs["reasoning_effort"] = analysis_reasoning_effort

analysis_lm = dspy.LM(**analysis_kwargs)

analysis_adapter = JSONAdapter()
hypothesis_adapter = JSONAdapter()

dspy.configure(lm=student_lm)
dspy.settings.configure(num_threads=num_threads)

/Users/aviram.kofman/Documents/Projects/dspy/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Dataset

Load AIME problems (American Invitational Mathematics Examination):

In [3]:
from datasets import load_dataset
import random

def init_dataset():
    train_split = load_dataset("AI-MO/aimo-validation-aime")['train']
    train_split = [
        dspy.Example({
            "problem": x['problem'],
            "solution": x['solution'],
            "answer": x['answer'],
        }).with_inputs("problem")
        for x in train_split
    ]
    
    random.Random(0).shuffle(train_split)
    tot_num = len(train_split)

    test_split = load_dataset("MathArena/aime_2025")['train']
    test_split = [
        dspy.Example({
            "problem": x['problem'],
            "answer": x['answer'],
        }).with_inputs("problem")
        for x in test_split
    ]

    train_set = train_split[: int(0.5 * tot_num)]
    val_set = train_split[int(0.5 * tot_num):]
    test_set = test_split * 5

    return train_set, val_set, test_set

train_set, val_set, test_set = init_dataset()

print(f"Training: {len(train_set)} | Validation: {len(val_set)} | Test: {len(test_set)}")

Training: 45 | Validation: 45 | Test: 150


Example problem:

In [4]:
print("Problem:", train_set[0]['problem'])
print("\nAnswer:", train_set[0]['answer'])

Problem: In isosceles trapezoid $ABCD$, parallel bases $\overline{AB}$ and $\overline{CD}$ have lengths $500$ and $650$, respectively, and $AD=BC=333$. The angle bisectors of $\angle{A}$ and $\angle{D}$ meet at $P$, and the angle bisectors of $\angle{B}$ and $\angle{C}$ meet at $Q$. Find $PQ$.

Answer: 242


## Program

Define a Chain of Thought program:

In [5]:
class GenerateResponse(dspy.Signature):
    """Solve the problem and provide the answer in the correct format."""
    problem = dspy.InputField()
    answer = dspy.OutputField()

program = dspy.ChainOfThought(GenerateResponse)

## Metrics

Define evaluation metrics:

In [6]:
def metric(example, prediction, trace=None, pred_name=None, pred_trace=None):
    correct_answer = int(example['answer'])
    try:
        llm_answer = int(prediction.answer)
    except ValueError:
        return 0
    return int(correct_answer == llm_answer)

In [7]:
def metric_with_feedback(example, prediction, trace=None, pred_name=None, pred_trace=None):
    correct_answer = int(example['answer'])
    written_solution = example.get('solution', '')
    
    try:
        llm_answer = int(prediction.answer)
    except ValueError:
        feedback_text = (
            f"The final answer must be a valid integer. "
            f"You responded with '{prediction.answer}', which couldn't be parsed. "
            f"The correct answer is '{correct_answer}'."
        )
        
        if written_solution:
            feedback_text += f" Here's the full solution:\n{written_solution}"
        
        return dspy.Prediction(score=0, feedback=feedback_text)

    score = int(correct_answer == llm_answer)
    
    if score == 1:
        feedback_text = f"Correct! The answer is '{correct_answer}'."
    else:
        feedback_text = f"Incorrect. The correct answer is '{correct_answer}'."

    if written_solution:
        feedback_text += f" Here's the full solution:\n{written_solution}"

    return dspy.Prediction(score=score, feedback=feedback_text)

## Baseline Evaluation

Evaluate the unoptimized program:

In [8]:
eval_kwargs = dict(
    num_threads=num_threads,
    display_progress=True,
    display_table=5,
    provide_traceback=False,
)

evaluate = dspy.Evaluate(
    devset=test_set,
    metric=metric,
    **eval_kwargs,
)

print("Evaluating baseline...")
baseline_result = evaluate(program)

print(f"\nBaseline Performance: {baseline_result.score:.1%}")

Evaluating baseline...
Average Metric: 80.00 / 150 (53.3%): 100%|██████████| 150/150 [00:00<00:00, 185.99it/s]

2025/10/14 20:15:03 INFO dspy.evaluate.evaluate: Average Metric: 80 / 150 (53.3%)


,problem,example_answer,reasoning,pred_answer,metric
0,Find the sum of all integer bases $b>9$ for which $17_b$ is a divi...,70,We interpret 17_b = b+7 and 97_b = 9b+7. We need b+7 to divide 9b+...,70,✔️ [1]
1,"On $\triangle ABC$ points $A, D, E$, and $B$ lie in that order on ...",588,"Set up affine coordinates with A=(0,0), B=(1,0), C=(0,1). Points o...",441,✔️ [0]
2,The 9 members of a baseball team went to an ice-cream parlor after...,16,"We must count assignments of 3 labeled flavors (C, V, S) to 9 dist...",16,✔️ [1]
3,"Find the number of ordered pairs $(x,y)$, where both $x$ and $y$ a...",117,"We need integer solutions (x,y) in [-100,100] satisfying 12x^2 - x...",117,✔️ [1]
4,There are $8!= 40320$ eight-digit positive integers that use each ...,279,Divisibility by 22 means divisible by 2 and 11. Units digit must b...,279,✔️ [1]



Baseline Performance: 5333.0%


## APEX Optimization

Optimize the program with APEX:

In [9]:
from dspy.teleprompt.apex_optimizer import APEX

optimizer = APEX(
    metric=metric_with_feedback,
    analysis_lm=analysis_lm,
    hypothesis_lm=analysis_lm,
    analysis_adapter=analysis_adapter,
    hypothesis_adapter=hypothesis_adapter,
    max_iterations=max_iterations,
    num_hypotheses=num_hypotheses,
    num_eval_runs=1,
    train_sample=train_sample_size,
    success_threshold=success_threshold,
    convergence_patience=convergence_patience,
    num_threads=num_threads,
    verbosity="normal",
    seed=seed,
    use_mlflow=use_mlflow,
    mlflow_tracking_uri=mlflow_tracking_uri,
    mlflow_experiment_name=mlflow_experiment_name,
)

print("Starting optimization...")
optimized_program = optimizer.compile(
    student=program,
    trainset=train_set,
    valset=val_set,
)

print("\nOptimization complete!")

2025/10/14 20:15:04 INFO dspy.teleprompt.apex_optimizer: APEX: MLflow tracking enabled


Starting optimization...


2025/10/14 20:15:04 INFO dspy.teleprompt.apex_optimizer: APEX: running with num_threads=50
2025/10/14 20:15:04 INFO dspy.teleprompt.apex_optimizer: APEX: Evaluating initial baseline on validation set


🏃 View run shivering-foal-447 at: http://localhost:5005/#/experiments/1/runs/7d9c9929f96145d0b172c575d1c2e794
🧪 View experiment at: http://localhost:5005/#/experiments/1
Processed 45 / 45 examples: 100%|██████████| 45/45 [00:00<00:00, 418.11it/s]

2025/10/14 20:15:04 INFO dspy.teleprompt.apex_optimizer: APEX: Initial baseline score=0.5111
2025/10/14 20:15:04 INFO dspy.teleprompt.apex_optimizer: APEX: iteration 1 started (train sample=20, val size=45)


...


Processed 18 / 20 examples:  95%|█████████▌| 19/20 [02:02<00:07,  7.07s/it]

2025/10/14 20:36:38 WARNING dspy.utils.parallelizer: SIGINT received. Cancelling.
2025/10/14 20:36:38 INFO dspy.teleprompt.apex_optimizer: APEX: Optimization interrupted by user (Ctrl+C)


Processed 18 / 20 examples:  95%|█████████▌| 19/20 [12:35<00:39, 39.78s/it]

2025/10/14 20:36:38 INFO dspy.teleprompt.apex_optimizer: APEX: Optimization complete - stopped after 2 iterations (interrupted)
2025/10/14 20:36:38 INFO dspy.teleprompt.apex_optimizer: APEX: Final score: 0.6889 (initial baseline: 0.5111)




Optimization complete!


Inspect the optimized prompt:

In [ ]:
print("Optimized Prompt:")
print("=" * 50)
print(optimized_program.predict.signature.instructions)
print("=" * 50)

## Final Evaluation

Evaluate the optimized program:

In [ ]:
print("Evaluating optimized program...")
optimized_result = evaluate(optimized_program)

print(f"\n{'='*50}")
print(f"Baseline:  {baseline_result.score:.1%}")
print(f"Optimized: {optimized_result.score:.1%}")
print(f"Improvement: {(optimized_result.score - baseline_result.score):.1%}")
print(f"{'='*50}")

## Optimization Insights

Examine the optimization process:

In [ ]:
if hasattr(optimized_program, 'apex_result'):
    result = optimized_program.apex_result
    
    print("Summary:")
    print(f"  Iterations: {len(result.iterations)}")
    print(f"  Candidates evaluated: {len(result.all_candidates)}")
    print(f"  Stop reason: {result.stopped_after}")
    print(f"  Best score: {result.best_candidate.overall_score:.4f}")
    
    print("\nIteration Progress:")
    for it in result.iterations:
        print(f"  Iteration {it.iteration}: {it.num_failures} failures, {len(it.hypotheses)} hypotheses, {len(it.candidates)} candidates")
    
    if result.best_candidate.hypothesis:
        h = result.best_candidate.hypothesis
        print(f"\nBest Hypothesis:")
        print(f"  Strategy: {h.strategy if hasattr(h, 'strategy') else 'N/A'}")
        print(f"  Impact Score: {h.impact_score if hasattr(h, 'impact_score') else 'N/A'}")

## MLflow Tracking (Optional)

If MLflow tracking was enabled, you can explore detailed logs:

```bash
mlflow ui --port 5000
```

Then navigate to `http://localhost:5000` to view:
- Detailed failure and success analyses
- All generated hypotheses with scores
- Per-example evaluation scores
- Complete optimization summary

## Conclusion

APEX systematically optimizes prompts through:
1. Analyzing failures to understand root causes
2. Recognizing successful patterns
3. Generating data-driven hypotheses
4. Validating improvements empirically

Try adjusting the configuration parameters to explore different optimization strategies.